# TFM: Generación Procedural de Contenido (PCG) en Videojuegos mediante Computación Cuántica y Optimización Clásica
### Demostración Interactiva del Pipeline Integral y Comparativa Experimental de Solvers

**Máster Universitario en Computación Cuántica**  
**Autora:** Belén Castañera  
**Director:** Paulet  

---

## 1. Contexto Metodológico y Arquitectura del Trabajo

Este notebook presenta una **demostración interactiva, rigurosa y reproducible** de la investigación desarrollada en el Trabajo de Fin de Máster. El objetivo central es investigar el diseño de formulaciones matemáticas, la escalabilidad y las propiedades del espacio de soluciones al aplicar **Computación Cuántica (QUBO, Recocido Cuántico y QAOA)** en simbiosis con **Optimización Clásica Exacta (CP-SAT)** a problemas de **Generación Procedural de Contenido (PCG)** en videojuegos.

---

### 1.1 Diagrama Metodológico del Flujo de Ejecución

El flujo del trabajo desacopla el modelado matemático, la validación lógica, la resolución algorítmica y la síntesis visual tridimensional:

```
   ┌────────────────────────────────────────────────────────┐
   │         PARÁMETROS DE ENTRADA AL PROCEDURAL            │
   │  (Dimensiones, saltos, densidad, coleccionables, etc.) │
   └───────────────────────────┬────────────────────────────┘
                               │
                               ▼
   ┌────────────────────────────────────────────────────────┐
   │         FORMULACIÓN MATEMÁTICA FORMAL (ILP/MIP)        │
   └─────────────┬───────────────────────────┬──────────────┘
                 │                           │
                 ▼                           ▼
   ┌───────────────────────────┐ ┌───────────────────────────┐
   │ VALIDACIÓN CLÁSICA EXACTA │ │     FORMULACIÓN QUBO      │
   │   (Google OR-Tools CP-SAT)│ │ (Matriz Q, offsets, penal)│
   │ [VERIFICACIÓN DETERMINISTA│ └─────────────┬─────────────┘
   │   DE ÓPTIMO / FACTIBILIDAD│               │
   └─────────────┬─────────────┘               │
                 │ Bucle de refinamiento       ▼
                 │ y verificación cotas   ┌─────────────────────────────────────────┐
                 └───────────────────────►│          RESOLUCIÓN Y MUESTREO          │
                                          │                                         │
                                          │  [Heurística Clásica sobre QUBO]        │
                                          │  • Simulated Annealing (D-Wave Ocean)   │
                                          │                                         │
                                          │  [Computación Cuántica]                 │
                                          │  • Quantum Annealing (QPU física)*      │
                                          │  • QAOA Variacional (Qiskit / Gate-based│
                                          └────────────────────┬────────────────────┘
                                                               │
                                                               ▼
   ┌────────────────────────────────────────────────────────┐
   │        VALIDACIÓN DE FACTIBILIDAD Y MÉTRICAS           │
   │     (Ausencia de atajos, TTS99, energía óptima)        │
   └───────────────────────────┬────────────────────────────┘
                               │
                               ▼
   ┌────────────────────────────────────────────────────────┐
   │              EXPORTACIÓN JSON NORMALIZADA              │
   │        (Geometría, coordenadas, rutas, entidades)      │
   └───────────────────────────┬────────────────────────────┘
                               │
                               ▼
   ┌────────────────────────────────────────────────────────┐
   │             BLENDER 3D (RENDERIZADO HEADLESS)          │
   │     (Texturas procedurales, monstruos, cofres, luces)  │
   └───────────────────────────┬────────────────────────────┘
                               │
                               ▼
   ┌────────────────────────────────────────────────────────┐
   │              ESCENA 3D Y DIORAMA FINAL                 │
   └────────────────────────────────────────────────────────┘
```
*\* Formulación matemáticamente apta para quantum annealing, sujeta a incrustación topológica (minor embedding) y restricciones de conectividad hardware.*

---

### 1.2 Justificación del Uso de CP-SAT: Solver Exacto Basado en Restricciones

En el desarrollo de este trabajo, el modelado del problema se construyó de manera evolutiva desde cero. En este contexto, el uso de **Google CP-SAT** desempeña un rol metodológico insustituible:
1. **Solver exacto basado en programación con restricciones:** CP-SAT combina propagación de dominios, aprendizaje de cláusulas en conflicto (CDCL proveniente de solvers SAT) y relajaciones lineales mediante branch-and-cut. **No realiza una búsqueda por fuerza bruta**, sino una poda analítica rigurosa capaz de **certificar deterministamente la optimalidad global** o **demostrar matemáticamente la infactibilidad** de una instancia.
2. **Necesidad de un oráculo de verdad terreno:** Si se utilizara directamente un solver heurístico o estocástico (como Simulated Annealing o una QPU física) y el algoritmo no obtuviera ninguna solución válida, resultaría imposible discernir si el fracaso proviene de un error matemático en la penalización del QUBO o de una limitación del solver estocástico al quedar atrapado en mínimos locales.
3. **Bucle de validación iterativo:** CP-SAT valida que las restricciones son consistentes, cerradas y factibles antes de trasladar el problema a una matriz QUBO para su análisis cuántico.

---

### 1.3 Entorno de Ejecución y Hardware Cuántico

* **Simulación y Emulación de Referencia:** Las evaluaciones de QAOA (algoritmo híbrido variacional cuántico-clásico) y de Simulated Annealing se ejecutan mediante simuladores clásicos de referencia controlados (*StatevectorSampler* de Qiskit y *SimulatedAnnealingSampler* de D-Wave Ocean).
* **Acceso a QPUs Físicas:** Para el desarrollo de este trabajo no se dispuso de acceso directo a una QPU comercial de recocido cuántico en tiempo de ejecución. La emulación clásica sobre CPU es el estándar metodológico en la literatura para aislar el comportamiento algorítmico, descartar el ruido térmico/de decoherencia y calcular de forma controlada el Time-to-Solution ($TTS$).
* **Aptitud Cuántica:** Todas las formulaciones QUBO desarrolladas son directamente aptas para recocido cuántico (*quantum annealing*), requiriendo en hardware físico un mapeo de grafo (*minor embedding*) adaptado a las topologías Pegasus o Zephyr de D-Wave.

## 2. Inicialización del Entorno y Verificación de Dependencias

In [ ]:
import sys
import os
import json
import time
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Añadir la raíz del repositorio y sus subpaquetes al path de Python
REPO_ROOT = Path(os.getcwd()).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Verificación de dependencias científicas
import dimod
import ortools
import qiskit
from common.metricas_tts import calcular_tts99

print("Entorno de ejecución inicializado con éxito:")
print(f" • Directorio raíz del proyecto: {REPO_ROOT}")
print(f" • Intérprete Python:            {sys.version.split()[0]}")
print(f" • Dimod (D-Wave Ocean SDK):     {dimod.__version__}")
print(f" • Google OR-Tools (CP-SAT):     {ortools.__version__}")
print(f" • Qiskit (Quantum SDK):         {qiskit.__version__}")
print(f" • Módulo unificado de métricas: common.metricas_tts (TTS99 discreto ceil activo)")

## 3. Caso 1: Colocación de Coleccionables ($p$-median)

En este escenario abordamos la colocación estratégica de $k$ monedas o recompensas sobre un grafo navegable $G=(V, E)$ para minimizar la distancia de acceso acumulada desde cualquier posición del jugador hasta la recompensa más cercana:
$$\min_{S \subseteq V, \, |S|=k} \sum_{i \in V} \min_{j \in S} d(i, j)$$

---

### Análisis de Codificación: Formulación Directa (20 Qubits) vs Formulación Compacta (4 Qubits)
* **Modelo Lineal Directo (20 qubits):**  
  Codifica $n=4$ variables de selección $x_j$ y $n^2=16$ variables de asignación $y_{ij}$. En este espacio de Hilbert de $2^{20} = 1.048.576$ estados, únicamente **96 estados son factibles (0.0092%)** y **6 estados son óptimos (0.00057%)**. En la simulación unitaria exacta del operador QAOA a $p=1$, la máxima probabilidad teórica de muestrear el óptimo es del $0.0099\%$, exigiendo más de **46.600 disparos** para garantizar un 99% de éxito. Con muestreos convencionales de 20 a 160 shots, la probabilidad empírica de observar el óptimo es $p_{\text{opt,emp}} = 0$, resultando en un **TTS no estimable** ($p_{\text{opt,emp}} = 0$).
* **Modelo Compacto para $k=2$ (4 qubits):**  
  Precalculando la matriz de costes combinatorios $C(j, l) = \sum_i \min(d(i, j), d(i, l))$, se eliminan todas las variables auxiliares $y_{ij}$. El problema opera estrictamente sobre $n=4$ variables binarias $x_j$, reduciendo el espacio de Hilbert a $2^4 = 16$ estados con una **densidad factible del 37.5%** y una **fracción óptima del 25.0%**. QAOA ($p=1$, COBYLA) muestrea el óptimo con una probabilidad del **65.625%**.

> **Nota Metodológica sobre Tiempos y TTS:**  
> En la tabla comparativa se distingue entre el **benchmark consolidado y congelado** en condiciones experimentales estabilizadas ($TTS_{99} = \mathbf{0.2872\text{ s}}$, documentado en `benchmark_qaoa_compacto_4q.json`) y el tiempo/TTS empírico de esta sesión viva puntual, el cual incluye el overhead de inicialización de Qiskit y la carga del entorno interactivo.

In [ ]:
from caso1_colocacion_monedas.mapas.mapa_qaoa_minimo import MAPA_QAOA_MINIMO
from caso1_colocacion_monedas.modelo.candidatas import obtener_candidatas
from caso1_colocacion_monedas.modelo.grafo import construir_grafo
from caso1_colocacion_monedas.modelo.distancias import construir_matriz_navegable
from caso1_colocacion_monedas.modelo.qubo_pmedian_compacto import (
    construir_qubo_pmedian_compacto,
    comprobar_factibilidad_compacto,
    coste_pmedian_compacto
)
from caso1_colocacion_monedas.solvers.k_medoids import k_medoids_pam
from caso1_colocacion_monedas.solvers.busqueda_exhaustiva import k_medoids_exhaustivo
from caso1_colocacion_monedas.solvers.simulated_annealing_qubo import resolver_qubo_simulated_annealing
from caso1_colocacion_monedas.solvers.qaoa_compacto import resolver_pmedian_qaoa_compacto

# Parámetros del experimento
K_COLECCIONABLES = 2
candidatas = obtener_candidatas(MAPA_QAOA_MINIMO)
grafo = construir_grafo(MAPA_QAOA_MINIMO)
matriz = construir_matriz_navegable(candidatas, grafo)

print(f"Instancia de prueba: {len(candidatas)} casillas candidatas, k={K_COLECCIONABLES} monedas")

# 1. Heurístico clásico (Partitioning Around Medoids - PAM)
t0 = time.perf_counter()
res_pam = k_medoids_pam(candidatas, matriz, K_COLECCIONABLES)
t_pam = time.perf_counter() - t0

# 2. Búsqueda exhaustiva exacta (Ground Truth matemático)
t0 = time.perf_counter()
res_exacta = k_medoids_exhaustivo(candidatas, matriz, K_COLECCIONABLES)
t_exacta = time.perf_counter() - t0
coste_optimo = res_exacta["coste_total"]

# 3. Formulación QUBO Compacta (fijando cota de penalización A con cota factible PAM)
qubo_compacto = construir_qubo_pmedian_compacto(matriz, k=K_COLECCIONABLES, cota_factible=res_pam["coste_total"])

# 4. Simulated Annealing clásico sobre QUBO (D-Wave Ocean)
t0 = time.perf_counter()
res_sa = resolver_qubo_simulated_annealing(qubo_compacto, num_reads=50, num_sweeps=100, seed=20260908)
t_sa = time.perf_counter() - t0
mejor_sa = res_sa["muestras"][0]
fact_sa = comprobar_factibilidad_compacto(qubo_compacto, mejor_sa["asignacion"])
coste_sa = coste_pmedian_compacto(matriz, fact_sa["seleccionadas"]) if fact_sa["factible"] else None
total_occ_sa = sum(m["num_occurrences"] for m in res_sa["muestras"])
optimos_sa = sum(m["num_occurrences"] for m in res_sa["muestras"] if abs(m["energia"] - mejor_sa["energia"]) < 1e-6)
p_sa_opt = optimos_sa / total_occ_sa
t_read_sa = t_sa / total_occ_sa
_, tts_sa = calcular_tts99(p_sa_opt, t_read_sa)

# 5. QAOA Compacto de 4 Qubits (p=1, COBYLA, 160 shots)
t0 = time.perf_counter()
res_qaoa = resolver_pmedian_qaoa_compacto(
    qubo_compacto, matriz, reps=1, maxiter=25, shots=160, seed=20260908, optimo_referencia=coste_optimo
)
t_qaoa = time.perf_counter() - t0
coste_qaoa = res_qaoa["coste"]
p_qaoa_opt = res_qaoa["analisis_muestras"]["probabilidad_optimo"]
p_qaoa_batch = 1.0 - (1.0 - p_qaoa_opt) ** res_qaoa["shots"]
_, tts_qaoa_sesion = calcular_tts99(p_qaoa_batch, t_qaoa)

# Benchmark consolidado de referencia en el TFM (condiciones estabilizadas)
TTS_QAOA_BENCHMARK_CONGELADO = 0.2872

print("\n" + "="*95)
print(f"{'Método / Solver':<30} | {'Coste':<6} | {'Óptimo':<7} | {'P(éxito)':<10} | {'TTS99 (s)':<18} | {'Tipo de Solver':<15}")
print("="*95)
print(f"{'1. PAM (Heurístico Clásico)':<30} | {res_pam['coste_total']:<6} | {str(res_pam['coste_total'] == coste_optimo):<7} | {'100.0%':<10} | {f'{t_pam:.5f} s':<18} | {'Clásico Heurístico':<15}")
print(f"{'2. Búsqueda Exhaustiva':<30} | {coste_optimo:<6} | {'True':<7} | {'100.0%':<10} | {f'{t_exacta:.5f} s':<18} | {'Clásico Exacto':<15}")
print(f"{'3. QUBO + Simulated Annealing':<30} | {coste_sa:<6} | {str(coste_sa == coste_optimo):<7} | {p_sa_opt*100:.1f}%     | {f'{tts_sa:.5f} s':<18} | {'Heurístico QUBO':<15}")
print(f"{'4. QAOA Compacto (4q, Sesión)':<30} | {coste_qaoa:<6} | {str(coste_qaoa == coste_optimo):<7} | {p_qaoa_opt*100:.1f}%     | {f'{tts_qaoa_sesion:.4f} s (sesión)':<18} | {'Variacional Cuántico':<15}")
print(f"{'   ↳ Ref. Benchmark Congelado':<30} | {coste_optimo:<6} | {'True':<7} | {'65.6%':<10} | {f'{TTS_QAOA_BENCHMARK_CONGELADO:.4f} s (ref.)':<18} | {'Consolidado TFM':<15}")
print(f"{'5. QAOA Directo (20q, Teórico)':<30} | {'-':<6} | {'False':<7} | {'0.0%':<10} | {'No estimable (p=0)':<18} | {'Variacional Cuántico':<15}")
print("="*95)

In [ ]:
# Visualización 2D de la Distribución de Monedas
fig, ax = plt.subplots(figsize=(7, 4))
filas = len(MAPA_QAOA_MINIMO)
cols = len(MAPA_QAOA_MINIMO[0])

# Dibujar celdas
for r in range(filas):
    for c in range(cols):
        char = MAPA_QAOA_MINIMO[r][c]
        color = '#2c3e50' if char == '#' else '#ecf0f1'
        ax.fill([c, c+1, c+1, c], [filas-r-1, filas-r-1, filas-r, filas-r], color=color, ec='#bdc3c7')

# Dibujar candidatas y elegidas por QAOA/Exacto
medoides_ids = {c["id"] for c in res_exacta["candidatas"]}
for idx, cand in enumerate(candidatas):
    r = cand["fila"]
    c = cand["columna"]
    y_plot = filas - r - 0.5
    x_plot = c + 0.5
    if cand["id"] in medoides_ids:
        ax.plot(x_plot, y_plot, 'o', color='#f39c12', markersize=18, mec='black', mew=2, label="Moneda Óptima" if idx == 0 else "")
        ax.text(x_plot, y_plot, "★", ha='center', va='center', color='white', fontsize=12, fontweight='bold')
    else:
        ax.plot(x_plot, y_plot, 's', color='#3498db', markersize=10, mec='black', label="Candidata libre" if idx == 1 else "")

ax.set_xlim(0, cols)
ax.set_ylim(0, filas)
ax.set_aspect('equal')
ax.set_title(f"Caso 1: Distribución Óptima de {K_COLECCIONABLES} Monedas (p-median, k={K_COLECCIONABLES})", fontsize=11, fontweight='bold')
ax.legend(loc='lower right')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Caso 2: Plataformas Jump & Run con Físicas Acotadas

En el Caso 2 se genera una secuencia de plataformas sobre un plano discretizado ($18 \times 5$) que garantiza un trayecto transitable entre un punto inicial $S=(0, 2)$ y una meta $G=(17, 2)$.

---

### Restricciones Cinemáticas y Evitación de Atajos
* **Físicas del Salto:** Se restringe la subida vertical máxima ($\Delta y \le 2$) y la caída vertical física ($\Delta y \ge -3$).
* **Condición Anti-Atajos:** El nivel generado debe garantizar que el camino previsto sea el camino más corto posible; no se admiten atajos directos que permitan al jugador saltarse plataformas intermedias.
* **QUBO Equivalente (5.504 variables):**  
  Modelar todas las restricciones cinemáticas y de flujo mediante variables binarias independientes genera un espacio de 5.504 variables y más de 576.000 coeficientes cuadráticos. Debido a las altas barreras de energía impuestas por las penalizaciones, el algoritmo de Simulated Annealing queda sistemáticamente atrapado en mínimos locales ($p_{\text{opt,emp}} = 0\%$, $TTS$ no estimable incluso con 10.000 sweeps).
* **QUBO Reducido ($C=3.0$):**  
  Al podar el grafo con físicas acotadas y penalizar con factor de conservación de flujo $C=3.0$, el modelo converge de manera altamente eficiente:
  * **Resultados Discretos Consolidados en la Memoria ($TTS_{99}$ con función ceil):**
    * **Factibilidad QUBO:** $TTS_{99}^{\text{fact}} \approx \mathbf{1.894\text{ s}}$ (sweeps = 50).
    * **Ruta Completa (Factible + Sin Atajos + $\le 2$ planos):** $TTS_{99}^{\text{completa}} \approx \mathbf{9.075\text{ s}}$ (sweeps = 50).

> **Aclaración sobre Coeficientes QUBO:**  
> La estructura de la matriz triangular contiene **551.775 coeficientes QUBO no nulos** con $C=3.0$. En la formulación base inicial con $C=1.0$ existían 542.687 coeficientes no nulos debido a cancelaciones algebraicas particulares que se rompen al modular el peso de flujo.

In [ ]:
from caso2_plataformas.modelo.grafo_saltos_segmentos_v4 import (
    ANCHO_PLATAFORMA, SUBIDA_MAX, CAIDA_MAX,
    obtener_anclas_candidatas, construir_grafo_segmentos_v4
)
import caso2_plataformas.cuantico.formulacion.qubo_caso2_18x5 as qmod2
from caso2_plataformas.cuantico.solvers.simulated_annealing import resolver_qubo_sa as resolver_sa_caso2

ANCHO = 18
ALTO = 5
START = (0, 2)
GOAL = (17, 2)

candidatas_c2 = obtener_anclas_candidatas(ANCHO, ALTO, START, GOAL)
posiciones_c2 = [START] + candidatas_c2 + [GOAL]
grafo_saltos = construir_grafo_segmentos_v4(posiciones_c2, START, GOAL)

print(f"Caso 2 (18x5): {len(candidatas_c2)} anclas candidatas | Start={START}, Goal={GOAL}")
print(f"Parámetros físicos cinemáticos: Subida máxima={SUBIDA_MAX}, Caída máxima={CAIDA_MAX}")

# Construir y resolver QUBO Reducido con factor de flujo C=3.0
qmod2.C = 3.0
t0 = time.perf_counter()
Q_c2, offset_c2 = qmod2.construir_qubo(grafo_saltos)
t_build = time.perf_counter() - t0

# Ejecutar Simulated Annealing con 50 sweeps (configuración de óptimo rendimiento demostrada)
res_sa_c2 = resolver_sa_caso2(Q_c2, num_reads=50, num_sweeps=50, seed=20260902)
muestra_opt = res_sa_c2.first.sample
energia_opt = res_sa_c2.first.energy + offset_c2

print(f"\nQUBO Reducido construido en {t_build:.4f} s:")
print(f" • Coeficientes QUBO no nulos en Q: {len(Q_c2)} (incluye diagonales y acoplamientos)")
print(f" • Energía mínima obtenida por SA:  {energia_opt:.2f} (Factible si <= 0.0)")
print(f" • Referencias discretas consolidadas en TFM: TTS_factible ≈ 1.894 s | TTS_completo ≈ 9.075 s")

In [ ]:
# Visualizar la trayectoria de plataformas generada
fig, ax = plt.subplots(figsize=(10, 3.5))

# Dibujar cuadrícula base
for x in range(ANCHO + 1):
    ax.axvline(x, color='#ecf0f1', lw=0.8)
for y in range(ALTO + 1):
    ax.axhline(y, color='#ecf0f1', lw=0.8)

# Dibujar anclas candidatas
for x, y in candidatas_c2:
    ax.plot(x, y, 'o', color='#bdc3c7', markersize=6)

# Dibujar Start y Goal
ax.plot(START[0], START[1], 's', color='#2ecc71', markersize=12, label="Inicio (Start)")
ax.plot(GOAL[0], GOAL[1], '*', color='#e74c3c', markersize=16, label="Meta (Goal)")

# Reconstruir ruta activa de la muestra
aristas_activas = [k for k, v in muestra_opt.items() if int(v) == 1]
for o, d in aristas_activas:
    ax.annotate("", xy=d, xytext=o,
                arrowprops=dict(arrowstyle="->", color="#8e44ad", lw=2.5, mutation_scale=15))
    # Plataforma como barra
    ax.plot([o[0]-0.4, o[0]+0.4], [o[1]-0.15, o[1]-0.15], color='#34495e', lw=4)

ax.set_xlim(-0.5, ANCHO + 0.5)
ax.set_ylim(-0.5, ALTO + 0.5)
ax.set_title("Caso 2: Plataformas y Trayectoria de Saltos Válida (QUBO Reducido C=3.0)", fontsize=11, fontweight='bold')
ax.set_xlabel("Coordenada X (Avance)")
ax.set_ylabel("Altura Y")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 5. Caso 3: Mazmorra 2D (Dungeon) con Rutas y Gameplay

En el Caso 3 se sintetiza proceduralmente una mazmorra completa sobre una cuadrícula de $6 \times 8$ (48 celdas).

---

### Clarificación Crucial sobre Restricciones Duras vs Métricas de Calidad
* **Restricción Dura de Ruta ($q_{t, c}$):** Se garantiza matemáticamente la existencia de una ruta navegable principal conexa entre $S=(0,0)$ y $G=(5,7)$ mediante variables paso-a-paso con continuidad ortogonal.
* **Conectividad Global del Suelo:** **No se modela como una restricción dura en el QUBO**, dado que requeriría variables adicionales de flujo multicommodity o cortes combinatorios de desconexión exponencial. La conectividad global es una **métrica de calidad emergente** favorecida por el acoplamiento ferromagnético tipo Ising ($\sum_{(u, v)} (x_u - x_v)^2$) y el balance zonal. La tasa empírica de conectividad perfecta observada supera el 90-95% en los muestreos.
* **Elementos de Gameplay y Rama Secundaria:**
  * 1 cofre de recompensa y 2 enemigos en la ruta principal.
  * 1 rama secundaria de exploración de 2 celdas con callejón sin salida estricto (*dead-end*) y un altar de recompensa secreta al final.

---

### Comparativa de Modelos QUBO en Caso 3: Integrado (96 vars) vs Completo (117 vars)
Es fundamental no confundir las métricas de ambos modelos consolidadas en el estudio de robustez (20 semillas independientes):
* **Modelo Integrado (96 variables):** Codifica geometría ($48\text{ vars } x_c$) y ruta ($48\text{ vars } q_{t, c}$).
  * Probabilidad de éxito / factibilidad: $\mathbf{93.60 \pm 2.35\%}$
  * Time-to-Solution ($TTS_{99}$): $\mathbf{4.39 \pm 0.60\text{ ms}}$
  * Longitud de fronteras suelo/pared: $22.65 \pm 0.81$
* **Modelo Completo / Full (117 variables):** Añade a lo anterior 9 variables de progreso de gameplay y 12 variables de selección de ramas secundarias ($b_k$).
  * Probabilidad de éxito / factibilidad: $\mathbf{54.35 \pm 4.17\%}$
  * Time-to-Solution ($TTS_{99}$): $\mathbf{19.47 \pm 5.06\text{ ms}}$
  * Longitud de fronteras suelo/pared: $22.95 \pm 0.76$

In [ ]:
from caso3_mapa.clasico.caso3_cpsat_v3 import solve_case3 as solve_cpsat_c3
from caso3_mapa.visualizacion.exportar_nivel_blender_caso3 import exportar_nivel as exportar_json_c3

SEMILLA = 42
print(f"Ejecutando resolución del Caso 3 con Semilla = {SEMILLA}...")

# 1. Resolver con Google CP-SAT v3 (Oráculo exacto basado en programación con restricciones)
t0 = time.perf_counter()
res_cpsat_c3 = solve_cpsat_c3(seed=SEMILLA)
t_cpsat_c3 = time.perf_counter() - t0

print(f"\nCP-SAT v3 resuelto en {t_cpsat_c3:.3f} s (Estado: {res_cpsat_c3['status']}):")
print(f" • Celdas transitables abiertas:  {len(res_cpsat_c3['open_cells'])} / 48")
print(f" • Longitud ruta principal:       {len(res_cpsat_c3['witness_path'])} pasos")
print(f" • Rama secundaria (2 celdas):    {res_cpsat_c3['branch_a']} -> {res_cpsat_c3['branch_b']}")
print(f" • Cofres en ruta principal:      {res_cpsat_c3['route_reward_cells']}")
print(f" • Enemigos en ruta principal:    {res_cpsat_c3['route_enemy_cells']}")
print(f" • Recompensa de rama secundaria: {res_cpsat_c3['branch_reward_cell']}")

# Exportar a formato JSON estructurado para el pipeline de Blender
ruta_json = exportar_json_c3(result=res_cpsat_c3, seed=SEMILLA)
print(f"\nArchivo JSON normalizado generado: {Path(ruta_json).name}")

In [ ]:
# Visualización 2D de la Mazmorra Generada con Gameplay
from caso3_mapa.clasico.caso3_cpsat_v3 import ROWS, COLS, START, GOAL

rows = ROWS
cols = COLS
open_cells = set(res_cpsat_c3["open_cells"])
path_cells = res_cpsat_c3["witness_path"]
branch_a = res_cpsat_c3["branch_a"]
branch_b = res_cpsat_c3["branch_b"]
branch_att = res_cpsat_c3["branch_attachment"]
branch_reward = res_cpsat_c3["branch_reward_cell"]
rewards = set(res_cpsat_c3["route_reward_cells"])
enemies = set(res_cpsat_c3["route_enemy_cells"])

fig, ax = plt.subplots(figsize=(9, 6))

# Dibujar celdas (Suelo vs Muro)
for r in range(rows):
    for c in range(cols):
        y = rows - r - 1
        if (r, c) in open_cells:
            ax.fill([c, c+1, c+1, c], [y, y, y+1, y], color='#ecf0f1', ec='#bdc3c7')
        else:
            ax.fill([c, c+1, c+1, c], [y, y, y+1, y], color='#2c3e50', ec='#1a252f')

# Dibujar Ruta Principal (línea verde continua)
pts_main_x = [c + 0.5 for r, c in path_cells]
pts_main_y = [rows - r - 0.5 for r, c in path_cells]
ax.plot(pts_main_x, pts_main_y, color='#27ae60', lw=4, zorder=3, label="Ruta Principal Garantizada")

# Dibujar Rama Secundaria (línea morada discontinua)
if branch_a and branch_b and branch_att:
    b_union = [branch_att, branch_a, branch_b]
    pts_b_x = [c + 0.5 for r, c in b_union]
    pts_b_y = [rows - r - 0.5 for r, c in b_union]
    ax.plot(pts_b_x, pts_b_y, color='#8e44ad', lw=3.5, linestyle='--', zorder=3, label="Rama Secreta (2 celdas)")

# Dibujar Entidades y Props
for r, c in rewards:
    ax.plot(c + 0.5, rows - r - 0.5, 'o', color='#f1c40f', markersize=14, mec='black', zorder=4, label="Cofre de Tesoro")
if branch_reward:
    ax.plot(branch_reward[1] + 0.5, rows - branch_reward[0] - 0.5, 'D', color='#e67e22', markersize=14, mec='black', zorder=4, label="Altar Secreto")
for r, c in enemies:
    ax.plot(c + 0.5, rows - r - 0.5, '^', color='#e74c3c', markersize=15, mec='black', zorder=4, label="Monstruo Demonio")

# Start y Goal
s_r, s_c = START
g_r, g_c = GOAL
ax.plot(s_c + 0.5, rows - s_r - 0.5, 's', color='#2ecc71', markersize=16, mec='black', zorder=5, label="Portal Inicio (Start)")
ax.plot(g_c + 0.5, rows - g_r - 0.5, 'p', color='#3498db', markersize=18, mec='black', zorder=5, label="Portal Salida (Goal)")

ax.set_xlim(0, cols)
ax.set_ylim(0, rows)
ax.set_aspect('equal')
ax.set_title(f"Caso 3: Mazmorra Procedural ({rows}x{cols}) con Gameplay y Rutas (CP-SAT v3)", fontsize=12, fontweight='bold')

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Pipeline Automatizado de Renderizado 3D en Blender

A partir del archivo JSON exportado por el solver, el script [`visualizar_caso3_blender.py`](file:///c:/Users/BCP/Desktop/TFM/caso3_mapa/visualizacion/visualizar_caso3_blender.py) construye de manera autónoma un diorama 3D completo en Blender:
* **Entorno y Diorama:** Plinth subterráneo de roca tallada, losas biseladas de piedra y cantería procedural con textura de relieve Bump 3D.
* **Modelos Procedurales 3D:**
  * **Monstruos demoníacos:** Malla orgánica con cuernos curvos, fauces con colmillos, garras y un gran ojo ciclópeo emisivo carmesí que proyecta luz local sobre el suelo.
  * **Cofres del tesoro:** Arcones de madera con refuerzos metálicos dorados, tapa abatida y acumulación de gemas brillantes.
  * **Portales y Antorchas:** Portales rúnicos emisivos para START/GOAL y antorchas con dispersión de luz puntual en muros interiores.
* **Renderizado Desatendido (Headless):** Se ejecuta con `blender -b -P ...` para generar la escena `.blend` y compilar el render fotorrealista en PNG.

In [ ]:
import subprocess
from IPython.display import Image, display

def localizar_blender():
    """Localiza el ejecutable de Blender en las rutas estándar de Windows."""
    rutas_posibles = [
        r"C:\Program Files\Blender Foundation\Blender 5.2\blender.exe",
        r"C:\Program Files\Blender Foundation\Blender 5.1\blender.exe",
        r"C:\Program Files\Blender Foundation\Blender 5.0\blender.exe",
        r"C:\Program Files\Blender Foundation\Blender 4.3\blender.exe",
    ]
    for ruta in rutas_posibles:
        if os.path.exists(ruta):
            return ruta
    return "blender"

def renderizar_nivel_en_blender(ruta_json_nivel, ejecutar_render=False):
    """Ejecuta Blender en segundo plano para generar la escena 3D y renderizar la imagen."""
    blender_exe = localizar_blender()
    script_blender = REPO_ROOT / "caso3_mapa" / "visualizacion" / "visualizar_caso3_blender.py"
    render_output = REPO_ROOT / "caso3_mapa" / "experimentos" / "figuras" / "caso3_blender_render_6x8.png"
    
    print(f"• Ejecutable Blender detectado: {blender_exe}")
    print(f"• Script de generación 3D:      {script_blender.name}")
    print(f"• Archivo JSON a procesar:      {Path(ruta_json_nivel).name}")
    
    if ejecutar_render:
        print("\nEjecutando Blender en segundo plano (headless)... Tiempo estimado: ~30 segundos...")
        t0 = time.perf_counter()
        
        env = os.environ.copy()
        env["CASO3_BLENDER_JSON"] = str(ruta_json_nivel)
        
        comando = [
            blender_exe,
            "-b",
            "-P", str(script_blender),
            "--", "--json", str(ruta_json_nivel)
        ]
        
        res = subprocess.run(comando, env=env, capture_output=True, text=True)
        t_render = time.perf_counter() - t0
        
        if res.returncode == 0:
            print(f"¡Renderizado 3D completado con éxito en {t_render:.2f} s!")
        else:
            print("Aviso en la ejecución de Blender. Detalle:")
            print(res.stderr[-500:])
            
    if os.path.exists(render_output):
        print(f"\nDesplegando render 3D fotorrealista ({render_output.name}):")
        display(Image(filename=str(render_output), width=850))
    else:
        print("Aviso: No se encontró el archivo renderizado PNG en disco.")

# Nota: Por defecto desplegamos directamente el render 3D de alta resolución.
# Puedes cambiar ejecutar_render=True para re-renderizar la escena en vivo con Blender en segundo plano (~30 s).
renderizar_nivel_en_blender(ruta_json, ejecutar_render=False)

## 7. Comparativa Global y Conclusiones del Trabajo

A continuación se sintetiza la evidencia cuantitativa de los tres casos de estudio, diferenciando con precisión la naturaleza de cada solver y las formulaciones exploradas:

| Caso de Estudio | Dimensión / Variables | Solver Clásico Exacto | Método / Muestreador sobre QUBO | Rendimiento Experimental ($TTS_{99}$) | Conclusión Metodológica Principal |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Caso 1: Monedas** | 4 qubits (compacto) vs 20 qubits (directo) | Búsqueda Exhaustiva / PAM | **QAOA Variacional** ($p=1$, COBYLA) y **Simulated Annealing** | **QAOA 4q:** $TTS = 0.2872\text{ s}$ (ref. congelada)<br>**QAOA 20q:** $TTS$ no estimable ($p_{\text{opt,emp}}=0$) | El precalculo matricial de asignaciones reduce el espacio en 5 órdenes de magnitud y hace viable el muestreo variacional. |
| **Caso 2: Plataformas** | 5.504 vars (equiv.) vs Reducido ($C=3.0$) | Google CP-SAT | **Simulated Annealing** (heurístico clásico) | **QUBO Reducido:**<br>$TTS_{\text{fact}} \approx 1.894\text{ s}$<br>$TTS_{\text{completa}} \approx 9.075\text{ s}$<br>**QUBO Equiv:** $TTS$ no estimable ($p_{\text{opt,emp}}=0$) | Los espacios QUBO masivos con altas penalizaciones crean paisajes rugosos que atrapan a los recocedores; la poda cinemática previa es indispensable. |
| **Caso 3: Mazmorra (Integrado)** | 96 variables (Geometría + Ruta) | Google CP-SAT v3 | **Simulated Annealing** (heurístico clásico) | Factibilidad: $93.60 \pm 2.35\%$<br>$TTS_{99} = 4.39 \pm 0.60\text{ ms}$ | La formulación desacoplada temporal de la ruta garantiza transitabilidad con altísima tasa de éxito. |
| **Caso 3: Mazmorra (Completo)** | 117 variables (Geom + Ruta + Gameplay + Rama) | Google CP-SAT v3 | **Simulated Annealing** (heurístico clásico) | Factibilidad: $54.35 \pm 4.17\%$<br>$TTS_{99} = 19.47 \pm 5.06\text{ ms}$ | Incorporar restricciones de branching y gameplay reduce la factibilidad al ~54%, pero mantiene un $TTS$ de milisegundos plenamente apto para PCG. |

---

### Conclusiones y Hallazgos Principales
1. **La Computación Cuántica en PCG exige formulaciones compactas:** La densidad de estados factibles en modelos ingenuos decrece exponencialmente. La reducción algebraica previa del espacio de búsqueda es el factor determinante para la viabilidad de algoritmos cuánticos como QAOA y Annealing.
2. **Sinergia Híbrida Indispensable:** Google CP-SAT proporciona la certeza matemática necesaria para validar cotas y certificar que las formulaciones son correctas antes de trasladarlas a QUBO, mientras que los modelos cuadráticos permiten su ejecución en arquitecturas cuánticas.
3. **Pipeline Modular Desacoplado:** El esquema $\text{Solver} \longrightarrow \text{JSON normalizado} \longrightarrow \text{Blender 3D}$ permite alternar libremente entre motores clásicos exactos, recocedores estocásticos y circuitos cuánticos variacionales sin alterar el motor visual final.